# 第10章 Canny边缘检测（跟随代码）

> 《OpenCV轻松入门：面向Python》（李立宗）第 10 章跟随代码。**全量收录原书代码**（例 10.1 + 章内独立演示），缺失例题已按书补全，代码以网页笔记 ch10.md 为准（原书素材 lena.bmp 按笔记统一换成 picture1.jpg）。
> `cv2.imshow` 需在**本地 Python 环境**运行（浏览器/Jupyter 内不弹窗）；运行前请确认**工作目录为素材根目录**（本文件夹，picture*.jpg/png 等素材都在这里），否则 `imread` 返回 None。
> 原 notebook 用到的 `cv_show` 是未给出定义的辅助函数，已在本 notebook 开头补上定义（本地运行显示窗口用）。


# 第十章 Canny边缘检测

> Canny 边缘检测是 1986 年 John F. Canny 提出的多级边缘检测算法，分四步：高斯滤波去噪 → 计算梯度的幅度与方向 → 非极大值抑制（把粗边缘"瘦"成单像素宽）→ 双阈值加滞后连接定夺最终边缘。核心思想是宁缺毋滥：只有证据足够强（梯度值超过高阈值）、或与强证据相连（滞后阈值思想）的点才算边缘。OpenCV 用一个函数 `cv2.Canny()` 完成去噪之外的全部流程。


In [ ]:
# cv_show 辅助函数定义（本地运行显示窗口用）：原 notebook 直接使用了它但未给出定义
def cv_show(name, img):
    cv2.imshow(name, img)
    cv2.waitKey(0)
    cv2.destroyAllWindows()

import cv2  # 供 cv_show 与后续示例使用


## 10.1 Canny 边缘检测基础：四步流程

1. **去噪**：噪声会影响边缘检测的准确性，通常先做高斯滤波——注意 `cv2.Canny()` 内部**不包含**这一步，要自己做（噪声大的图 Canny 前加一行 `GaussianBlur` 是标准动作）；核越大越抗噪，但定位误差也随之增加，5×5 的核通常够用。
2. **计算梯度的幅度与方向**：幅度 G = √(Gx²+Gy²)，方向 Θ = atan2(Gy, Gx)；**梯度方向与边缘方向垂直**，就近取 8 个方向之一。
3. **非极大值抑制**：沿正/负梯度方向判断当前点是否为局部最大值——是则保留，否则抑制归零；同一方向基本只留一个点，完成边缘细化。
4. **双阈值确定边缘**：梯度值 ≥ maxVal 为强边缘；≤ minVal 抑制；介于两者之间为虚边缘（弱边缘）——与强边缘相连的保留，不相连的抛弃。这套"双阈值 + 连接判断"的机制学名叫滞后阈值法（hysteresis thresholding）。


## 10.2 Canny 函数及使用

`edges=cv2.Canny(img,threshold1,threshold2,apertureSize,L2gradient)`

- `img`：输入图像，8 位图像（例题按 `IMREAD_GRAYSCALE` 读成单通道灰度图）。
- `threshold1` / `threshold2`：两个阈值。**顺序无关**——OpenCV 内部取较小者作 minVal（边缘连接）、较大者作 maxVal（寻找强边缘初始片段）；阈值越小捕获的边缘越多，高低调之比建议 2:1~3:1，且不是固定值，要针对不同图像调整。
- `apertureSize`：Sobel 算子的孔径大小（即第 9 章 Sobel 的 ksize），默认为 3。
- `L2gradient`：计算梯度幅度的标识，默认 False 用 L1 范数（两方向导数绝对值相加），True 用 L2 范数（平方和开方，更精确）。

**代码导览**：例 10.1（不同阈值组合的边缘检测）→ 独立演示（自选阈值对比 + 双滑动条交互调参）。


### 例 10.1 · 不同阈值组合的边缘检测

使用 `cv2.Canny()` 获取图像边缘，并尝试不同的 threshold1、threshold2 组合：阈值 (128, 200) 的边缘稀疏，(32, 128) 捕获更多边缘信息——**threshold1 和 threshold2 的值越小，能够捕获的边缘信息越多**。（原书 lena.bmp 按笔记换成 picture1.jpg。）


In [ ]:
# ===== 10.2 例 10.1：不同阈值组合的 Canny 边缘检测 =====
import cv2

o = cv2.imread("picture1.jpg", cv2.IMREAD_GRAYSCALE)  # 灰度图读入（原书 lena.bmp）
r1 = cv2.Canny(o, 128, 200)   # 高阈值组合：threshold1=128，threshold2=200
r2 = cv2.Canny(o, 32, 128)    # 低阈值组合：threshold1=32， threshold2=128

cv2.imshow("original", o)     # (a) 原始图像
cv2.imshow("result1", r1)     # (b) 高阈值组合的边缘检测结果
cv2.imshow("result2", r2)     # (c) 低阈值组合的边缘检测结果
cv2.waitKey()
cv2.destroyAllWindows()


#### 独立演示 · 自选阈值对比与双滑动条交互调参（picture2.png）

前半段用自有素材 picture2.png 对比 (128, 255) 与 (32, 96) 两组阈值（例 10.1 的自有素材版）；后半段搭一个双滑动条窗口，拖动 Min/Max Threshold 实时观察 Canny 结果，按 Esc（键值 27）退出——两个阈值不是固定值，实践中常用滑动条交互调参。


In [243]:
img=cv2.imread("picture2.png",cv2.IMREAD_GRAYSCALE)

edges_1=cv2.Canny(img,128,255)
edges_2=cv2.Canny(img,32,96)

cv_show("Original Image",img)
cv_show("Edges 1",edges_1)
cv_show("Edges 2",edges_2)


#自动寻找最佳阈值
def everything_is_edge(x):
    #回调函数，什么都不做
    pass

img=cv2.imread("picture2.png",cv2.IMREAD_GRAYSCALE)

if img is None:

    img=np.zeros((512,512),np.uint8)
    cv2.circle(img,(256,256),100,(255),-1)
    cv2.circle(img,(256,256),50,(0),-1)

cv2.namedWindow("Canny Turning")

#创建两个滑动条
cv2.createTrackbar("Min Threshold","Canny Turning",0,255,everything_is_edge)
cv2.createTrackbar("Max Threshold","Canny Turning",0,255,everything_is_edge)

while True:
    #获取滑动条的值
    min_val=cv2.getTrackbarPos("Min Threshold","Canny Turning")
    max_val=cv2.getTrackbarPos("Max Threshold","Canny Turning")

    #进行Canny边缘检测
    edges=cv2.Canny(img,min_val,max_val)

    #显示结果
    cv2.imshow("Canny Turning",edges)

    k=cv2.waitKey(0)
    if k==27:
        break

cv2.destroyAllWindows()